# 신용카드 고객 이탈 예측 프로젝트
##  데이터 로드 + 정제
데이터: `BankChurners.csv` (Kaggle "Credit Card Customers")

## 1. 데이터 로드

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/BankChurners.csv")

print("원본 shape:", df.shape)   # (10127, 23)
df.head()

원본 shape: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


 데이터 사전 (Data Dictionary)
|컬럼명| 설명| 타입| 비고|
|---|---|---|---|
|CLIENTNUM |고객 고유 식별번호 |식별자| 모델 변수에서제외|
|Attrition_Flag| 이탈 여부| (Existing Customer /Attrited Customer)|범주형| 타겟 변수|
|Customer_Age| 고객| 나이| 수치형|
|Gender| 성별| (M/F)| 범주형|
|Dependent_count |부양가족 수 |수치형|---|
|Education_Level |학력(High School, Graduate,Doctorate 등)|범주형| 결측값 "Unknown"포함|
|Marital_Status |결혼상태 (Married, Single,Divorced 등)|범주형 |결측값 "Unknown"포함|
|Income_Category| 소득 구간 ($40K 미만 ~ $120K이상)|범주형| 결측값 "Unknown"포함|
|Card_Category| 카드 등급| (Blue, Silver, Gold,Platinum)|범주형| 대부분 Blue에편중|
|Months_on_book |카드 보유 개월 수 |수치형| 생존분석 duration 변수로 활용 가능|
|Total_Relationship_Count| 보유 상품(계좌·서비스) 개수 |수치형 |관계 깊이 지표|
|Months_Inactive_12_mon| 최근 12개월 중 비활성 개월 수 |수치형 |휴면 위험 핵심 지표|
|Contacts_Count_12_mon | 최근 12개월 고객센터 문의 횟수 |수치형| 불만 누적 신호|
|Credit_Limit |신용한도| 수치형|---|
|Total_Revolving_Bal |리볼빙(이월) 잔액 |수치형|---|
|Avg_Open_To_Buy| 평균 이용 가능 한도 (한도 -리볼빙잔액)|수치형|---|
|Total_Amt_Chng_Q4_Q1 |1분기 대비 4분기 거래금액 변화율 |수치형| 급감 시 이탈 전조 가능|
|Total_Trans_Amt |최근 12개월 총 거래금액 |수치형 |고객가치(LTV)대리 지표|
|Total_Trans_Ct |최근 12개월 총 거래건수 |수치형| 핵심 예측 변수로|
|Total_Ct_Chng_Q4_Q1| 1분기 대비 4분기 거래건 수 변화율 |수치형| 핵심 예측 변수|
|Avg_Utilization_Ratio| 평균 한도소진 율(리볼빙잔액/한도) |수치형 (0~1)| 재정 압박 또는 활발한 이용 양방향 해석 가능|
|Naive_Bayes_Classifier_...1| 나이브베이즈 예측 확률값 1 |수치형| 데이터 누출컬럼 — 반드시제거|
|Naive_Bayes_Classifier_...2 |나이브베이즈 예측 확률값 2 |수치형 | 데이터 누출컬럼 — 반드시 제거|

## 2. 불필요/누출 컬럼 제거
- **CLIENTNUM**: 단순 식별자, 예측에 사용하지 않음
- **Naive_Bayes_Classifier_...** (2개): 타겟 정보가 이미 반영된 데이터 누출 컬럼 → 반드시 제거

In [2]:
df.isna().sum()

CLIENTNUM                                                                                                                             0
Attrition_Flag                                                                                                                        0
Customer_Age                                                                                                                          0
Gender                                                                                                                                0
Dependent_count                                                                                                                       0
Education_Level                                                                                                                       0
Marital_Status                                                                                                                        0
Income_Category                                 

In [3]:
df.columns

Index(['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'],
      dtype='str')

In [17]:
print(df['Total_Amt_Chng_Q4_Q1'].sort_values()) # 1분기 대비 4분기 거래금액 변화율
print(df['Total_Ct_Chng_Q4_Q1'].sort_values())  # 1분기 대비 4분기 거래건수 변화율

4417    0.000
7998    0.000
7207    0.000
4701    0.000
3596    0.000
        ...  
219     2.368
2       2.594
773     2.675
8       3.355
12      3.397
Name: Total_Amt_Chng_Q4_Q1, Length: 10127, dtype: float64
7998    0.000
4417    0.000
7165    0.000
1905    0.000
3596    0.000
        ...  
113     3.000
12      3.250
269     3.500
773     3.571
1       3.714
Name: Total_Ct_Chng_Q4_Q1, Length: 10127, dtype: float64


In [4]:
cols_drop = [
    "CLIENTNUM",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]
df = df.drop(columns=cols_drop)

print("컬럼 제거 후 shape:", df.shape)   # (10127, 20)

컬럼 제거 후 shape: (10127, 20)


## 3. 결측치 확인
실제 `NaN`은 없지만, `Education_Level` / `Marital_Status` / `Income_Category`에는 "Unknown"이라는 문자열로 결측이 표시되어 있어 실질적 결측치로 간주해야 합니다.

이번 **기초 단계**에서는 Unknown을 별도 카테고리로 그대로 유지합니다 (제거·대체는 이후 고도화 단계에서 검토).

In [5]:
print(df['Education_Level'].unique())
print(df['Marital_Status'].unique())
print(df['Income_Category'].unique())

<StringArray>
[  'High School',      'Graduate',    'Uneducated',       'Unknown',
       'College', 'Post-Graduate',     'Doctorate']
Length: 7, dtype: str
<StringArray>
['Married', 'Single', 'Unknown', 'Divorced']
Length: 4, dtype: str
<StringArray>
[   '$60K - $80K', 'Less than $40K',   '$80K - $120K',    '$40K - $60K',
        '$120K +',        'Unknown']
Length: 6, dtype: str


In [6]:
print("실제 NaN 개수:", df.isna().sum().sum())

unknown_cols = ["Education_Level", "Marital_Status", "Income_Category"]
for col in unknown_cols:
    n_unknown = (df[col] == "Unknown").sum()
    print(f"{col} - Unknown 비율: {n_unknown/len(df)*100:.1f}% ({n_unknown}건)")

실제 NaN 개수: 0
Education_Level - Unknown 비율: 15.0% (1519건)
Marital_Status - Unknown 비율: 7.4% (749건)
Income_Category - Unknown 비율: 11.0% (1112건)


## 4. 타겟 변수 인코딩
`Attrition_Flag`: "Existing Customer" / "Attrited Customer" → 0 / 1 (`Target`)

In [7]:
df["Target"] = df["Attrition_Flag"].map(
    {"Existing Customer": 0, "Attrited Customer": 1}
)
df = df.drop(columns=["Attrition_Flag"])


In [8]:
df["Target"].value_counts()

Target
0    8500
1    1627
Name: count, dtype: int64

## 5. 이상치 간단 확인 (수치형 컬럼 요약통계)

In [9]:
numeric_cols = df.select_dtypes(include="number").columns.drop("Target")
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Customer_Age,10127.0,46.325960,8.016814,26.0,41.000,46.000,52.000,73.000
Dependent_count,10127.0,2.346203,1.298908,0.0,1.000,2.000,3.000,5.000
Months_on_book,10127.0,35.928409,7.986416,13.0,31.000,36.000,40.000,56.000
Total_Relationship_Count,10127.0,3.812580,1.554408,1.0,3.000,4.000,5.000,6.000
Months_Inactive_12_mon,10127.0,2.341167,1.010622,0.0,2.000,2.000,3.000,6.000
Contacts_Count_12_mon,10127.0,2.455317,1.106225,0.0,2.000,2.000,3.000,6.000
Credit_Limit,10127.0,8631.953698,9088.776650,1438.3,2555.000,4549.000,11067.500,34516.000
Total_Revolving_Bal,10127.0,1162.814061,814.987335,0.0,359.000,1276.000,1784.000,2517.000
Avg_Open_To_Buy,10127.0,7469.139637,9090.685324,3.0,1324.500,3474.000,9859.000,34516.000
Total_Amt_Chng_Q4_Q1,10127.0,0.759941,0.219207,0.0,0.631,0.736,0.859,3.397


## 6. 정제된 데이터 저장

In [10]:
df.to_csv("../data/processed/bankchurners_clean.csv", index=False)


print(f"최종: {df.shape}")

최종: (10127, 20)
